# Assignment 1: Linear regression implementation

This notebook repeats the reference workflow: inspect and plot the data, compare a closed-form least-squares solution with scikit-learn, implement gradient descent, examine convergence under different learning rates and iteration counts, and compare test RMS across model feature sets. The train/test split is deterministic (`random_state=42`). Test/train ratios are 0.25, 0.5, and 1.0.

**Metric:** RMS = square root of the mean squared prediction error on held-out rows. Coefficients are reported as `y = X @ w + b`.


### Dataset source

The included `cardekho_used_cars.csv` is the `CAR DETAILS FROM CAR DEKHO.csv` file from the [Kaggle Vehicle dataset](https://www.kaggle.com/datasets/nehalbirla/vehicle-dataset-from-cardekho). Kaggle requires sign-in for direct download; the included file was obtained from a public [GitHub mirror](https://github.com/bagassenop/cardekho/blob/main/CAR%20DETAILS%20FROM%20CAR%20DEKHO.csv). See `DATA_LICENSE.md` for attribution and database-license information.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42


## Data and model definitions

Kaggle CarDekho: used-car price prediction. The three requested feature sets are compared below, using manufacturing year and kilometers driven to predict selling price in INR.

In [ ]:
# Read the included CSV. All required data files are stored beside this notebook.
df = pd.read_csv("cardekho_used_cars.csv")
# Remove an exported row index column, if present.
df = df.loc[:, ~df.columns.astype(str).str.match(r"^Unnamed")]

# Add polynomial terms used by the requested models.
df["km_driven_sq"] = df["km_driven"] ** 2
target = "selling_price"
feature_sets = {
    "a": ['year'],
    "b": ['year', 'km_driven'],
    "c": ['year', 'km_driven', 'km_driven_sq'],
}
required = sorted({target, *[feature for features in feature_sets.values() for feature in features]})
df = df.dropna(subset=required).reset_index(drop=True)
print(f"Rows used: {len(df)}")
display(df.head())
display(df[required].describe().round(2))


## Visualize the data

In [ ]:
# Plot the target against each base predictor; this makes scale and trend visible.
base_features = ['year']
fig, axes = plt.subplots(1, len(base_features), figsize=(5 * len(base_features), 4))
if len(base_features) == 1:
    axes = [axes]
for ax, feature in zip(axes, base_features):
    ax.scatter(df[feature], df[target], alpha=0.75)
    ax.set_xlabel(feature)
    ax.set_ylabel(target)
    ax.set_title(f"{target} vs {feature}")
plt.tight_layout()
plt.show()


## Fit coefficients and compare implementations

The first fit uses all available rows to report each model’s final coefficients. The next cell independently fits the first model using scikit-learn and compares coefficients and training RMS.

In [ ]:
def fit_closed_form(X, y):
    """Least-squares fit with an explicit bias column; returns weights and bias."""
    design = np.column_stack([np.asarray(X, dtype=float), np.ones(len(y))])
    theta, *_ = np.linalg.lstsq(design, np.asarray(y, dtype=float), rcond=None)
    return theta[:-1], float(theta[-1])

full_fits = {}
for name, features in feature_sets.items():
    X = df[features].to_numpy(dtype=float)
    y = df[target].to_numpy(dtype=float)
    w, bias = fit_closed_form(X, y)
    full_fits[name] = (w, bias)
    print(f"Model {name}: features={features}")
    print(f"  w={w}; b={bias:.8f}")
    print(f"  equation: {target} = " + " + ".join(
        [f"({coef:.8g} * {feature})" for coef, feature in zip(w, features)] + [f"({bias:.8g})"]
    ))
    print()


## Reference exercise: one-feature cost surface

In [ ]:
# Reference exercise: visualize the MSE surface for the first, one-feature model.
# The grid is centered on the closed-form solution so the minimum is visible.
x_cost = df[feature_sets["a"][0]].to_numpy(dtype=float)
y_cost = df[target].to_numpy(dtype=float)
w_star, b_star = fit_closed_form(x_cost.reshape(-1, 1), y_cost)
w_grid = np.linspace(w_star[0] - max(abs(w_star[0]) * 0.5, 1.0), w_star[0] + max(abs(w_star[0]) * 0.5, 1.0), 45)
b_grid = np.linspace(b_star - max(np.std(y_cost), 1.0), b_star + max(np.std(y_cost), 1.0), 45)
W, B = np.meshgrid(w_grid, b_grid)
Z = np.mean((W[..., None] * x_cost + B[..., None] - y_cost) ** 2, axis=2)
fig = plt.figure(figsize=(8, 5))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(W, B, Z, cmap="viridis", alpha=0.85)
ax.set_xlabel("w")
ax.set_ylabel("b")
ax.set_zlabel("MSE")
ax.set_title("One-feature MSE cost surface")
plt.tight_layout()
plt.show()


In [ ]:
# Verify the unregularized least-squares result against scikit-learn.
features = feature_sets["a"]
X = df[features].to_numpy(dtype=float)
y = df[target].to_numpy(dtype=float)
w, bias = fit_closed_form(X, y)
reference = LinearRegression().fit(X, y)
print("NumPy w,b:", w, bias)
print("sklearn w,b:", reference.coef_, reference.intercept_)
print("Maximum coefficient difference:", np.max(np.abs(w - reference.coef_)))
print("Training RMS:", np.sqrt(np.mean((y - reference.predict(X)) ** 2)))


## Held-out RMS at different test/train ratios

For ratio r, the test fraction is r / (1 + r). All feature sets use the same deterministic split for a given ratio.

In [ ]:
rows = []
for ratio in (0.25, 0.5, 1.0):
    test_fraction = ratio / (1.0 + ratio)
    for name, features in feature_sets.items():
        X = df[features].to_numpy(dtype=float)
        y = df[target].to_numpy(dtype=float)
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=test_fraction, random_state=RANDOM_STATE
        )
        model = LinearRegression().fit(X_train, y_train)
        predictions = model.predict(X_test)
        rows.append({
            "model": name,
            "features": ", ".join(features),
            "test/train ratio": ratio,
            "train rows": len(y_train),
            "test rows": len(y_test),
            "test RMS": np.sqrt(np.mean((y_test - predictions) ** 2)),
        })
rms_results = pd.DataFrame(rows)
display(rms_results.pivot(index="test/train ratio", columns="model", values="test RMS").round(3))
display(rms_results.round(3))


## Reference experiment: polynomial degree

In [ ]:
# Reference exercise: compare polynomial degree on one predictor.
# Reuse the largest test fraction (test/train = 1) for this degree sweep.
x_degree = df[["km_driven"]].to_numpy(dtype=float)
y_degree = df[target].to_numpy(dtype=float)
X_train, X_test, y_train, y_test = train_test_split(
    x_degree, y_degree, test_size=0.5, random_state=RANDOM_STATE
)
degree_rows = []
for degree in range(1, 6):
    Xtr = np.column_stack([X_train[:, 0] ** power for power in range(1, degree + 1)])
    Xte = np.column_stack([X_test[:, 0] ** power for power in range(1, degree + 1)])
    model = LinearRegression().fit(Xtr, y_train)
    degree_rows.append({
        "degree": degree,
        "train RMS": np.sqrt(np.mean((y_train - model.predict(Xtr)) ** 2)),
        "test RMS": np.sqrt(np.mean((y_test - model.predict(Xte)) ** 2)),
    })
degree_results = pd.DataFrame(degree_rows)
display(degree_results.round(3))
plt.plot(degree_results["degree"], degree_results["train RMS"], "o-", label="train")
plt.plot(degree_results["degree"], degree_results["test RMS"], "o-", label="test")
plt.xlabel("Polynomial degree")
plt.ylabel("RMS")
plt.title("Polynomial degree experiment")
plt.legend()
plt.show()


## Reference experiment: gradient descent and feature scaling

Gradient descent is run after standardizing each input feature and the target. We compare learning rates and iteration budgets. Standardization keeps differently scaled predictors on comparable numerical ranges.

In [ ]:
def gradient_descent(X, y, learning_rate, iterations, initial_w=None, initial_b=0.0):
    """Batch gradient descent for mean squared error on standardized data."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float).reshape(-1)
    m, n = X.shape
    w = np.zeros(n) if initial_w is None else np.asarray(initial_w, dtype=float).copy()
    bias = float(initial_b)
    history = []
    for step in range(iterations):
        error = X @ w + bias - y
        dw = (X.T @ error) / m
        db = error.mean()
        w -= learning_rate * dw
        bias -= learning_rate * db
        if step % max(1, iterations // 100) == 0:
            history.append(float(np.mean((X @ w + bias - y) ** 2)))
    return w, bias, history

# Demonstrate sensitivity to initialization, step size, and iteration budget.
features = feature_sets["c"]
X = df[features].to_numpy(dtype=float)
y = df[target].to_numpy(dtype=float)
x_scaler = StandardScaler().fit(X)
y_scaler = StandardScaler().fit(y.reshape(-1, 1))
X_scaled = x_scaler.transform(X)
y_scaled = y_scaler.transform(y.reshape(-1, 1)).ravel()
experiment_rows = []
for rate in (0.01, 0.1, 0.5):
    for iterations in (100, 1000):
        w0 = np.full(X_scaled.shape[1], 0.1)  # nonzero initialization experiment
        w_gd, b_gd, history = gradient_descent(
            X_scaled, y_scaled, learning_rate=rate, iterations=iterations, initial_w=w0
        )
        pred_scaled = X_scaled @ w_gd + b_gd
        pred = y_scaler.inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
        experiment_rows.append({
            "learning rate": rate,
            "iterations": iterations,
            "final scaled MSE": history[-1],
            "full-data RMS (original units)": np.sqrt(np.mean((y - pred) ** 2)),
        })
display(pd.DataFrame(experiment_rows).round(5))


## Conclusion

Model (a), which uses year alone, has the lowest held-out RMS at all three test/train ratios. Adding kilometers driven barely changes model (b) error, while the squared-mileage model (c) performs worse on each split. The nonlinear term adds flexibility but appears to overfit or behave unstably on these splits; more predictors do not guarantee better generalization. These comparisons use a fixed random seed and reflect one dataset and split procedure.

Selling prices are reported in INR in the source data. Coefficients describe this dataset and do not establish causal effects.

### Full-data fitted coefficients

| Model | Predictors | Weights w (in listed order) | Bias b |
|---|---|---|---|
| a | year | 56810.04 | -1.138596e+08 |
| b | year, km_driven | 55511.65, -0.2795857 | -1.112274e+08 |
| c | year, km_driven, km_driven_sq | 365.5473, -4.192427, 7.046328e-06 | 0.1812601 |

### Held-out test RMS

| Model | Test/train ratio | Train n | Test n | RMS (INR) |
|---|---:|---:|---:|---:|
| a | 0.25 | 3472 | 868 | 505615.851 |
| a | 0.5 | 2893 | 1447 | 529739.761 |
| a | 1 | 2170 | 2170 | 515117.502 |
| b | 0.25 | 3472 | 868 | 505675.122 |
| b | 0.5 | 2893 | 1447 | 529837.120 |
| b | 1 | 2170 | 2170 | 515442.635 |
| c | 0.25 | 3472 | 868 | 537804.989 |
| c | 0.5 | 2893 | 1447 | 565215.044 |
| c | 1 | 2170 | 2170 | 551292.051 |